# Fase 2 — Notebook 02: Otimização com Algoritmo Genético

**Tech Challenge FIAP — VRP para Saúde da Mulher**

Este notebook executa e analisa os 3 experimentos do Algoritmo Genético:

| Experimento | Pop | Mutação | Gerações |
|:-----------:|:---:|:-------:|:--------:|
| 1 | 50  | 0.10 | 30 |
| 2 | 100 | 0.05 | 50 |
| 3 | 200 | 0.15 | 50 |

Análise inclui:
- Curvas de convergência
- Decomposição da função fitness
- Visualização das rotas otimizadas
- Comparação estatística entre experimentos

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent.parent))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import json

from fase2_vrp.src.data_generator import generate_service_points, get_depot, build_distance_matrix
from fase2_vrp.src.genetic_algorithm import GAConfig, run_genetic_algorithm
from fase2_vrp.src.fitness import compute_fitness, fitness_breakdown
from fase2_vrp.src.constraints import _route_distance, compute_arrival_times
from fase2_vrp.src.route_visualizer import create_route_map, create_comparison_map

plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['font.size'] = 11
sns.set_style('whitegrid')

RESULTS_DIR = Path('../results')
RESULTS_DIR.mkdir(exist_ok=True)

print('Setup completo!')

In [ ]:
# Gerar dados
SEED = 42
depot  = get_depot()
points = generate_service_points(n_points=20, seed=SEED)
dist_matrix = build_distance_matrix(depot, points)
print(f'Dados gerados: {len(points)} pontos de atendimento')

In [ ]:
# Configurações dos 3 experimentos
EXPERIMENTS = [
    ('Exp 1 — Pop=50  | Mut=0.10 | 30 gen',
     GAConfig(population_size=50,  n_generations=30, mutation_rate=0.10, seed=SEED)),
    ('Exp 2 — Pop=100 | Mut=0.05 | 50 gen',
     GAConfig(population_size=100, n_generations=50, mutation_rate=0.05, seed=SEED)),
    ('Exp 3 — Pop=200 | Mut=0.15 | 50 gen',
     GAConfig(population_size=200, n_generations=50, mutation_rate=0.15, seed=SEED)),
]

print('Configurações:')
for label, cfg in EXPERIMENTS:
    print(f'  {label}')

In [ ]:
# Executar os 3 experimentos
results = []
for label, config in EXPERIMENTS:
    print(f'\n>>> {label}')
    result = run_genetic_algorithm(points, dist_matrix, config, verbose=True)
    results.append((label, result))

print('\n✓ Todos os experimentos concluídos!')

In [ ]:
# Tabela comparativa
rows = []
for label, res in results:
    route = res.best_individual.chromosome
    bd = fitness_breakdown(route, points, dist_matrix)
    rows.append({
        'Experimento': label,
        'Fitness Inicial': round(res.best_fitness_hist[0], 2),
        'Fitness Final': round(res.best_individual.fitness, 2),
        'Melhoria (%)': round((1 - res.best_individual.fitness / res.best_fitness_hist[0]) * 100, 1),
        'Distância (km)': round(bd['distance'], 1),
        'Pen. Prioridade': round(bd['penalty_priority'], 1),
        'Pen. Capacidade': round(bd['penalty_capacity'], 1),
        'Pen. Tempo': round(bd['penalty_time'], 1),
        'Tempo (s)': res.execution_time_s,
    })

df_results = pd.DataFrame(rows)
df_results

In [ ]:
# Curvas de convergência
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
colors = ['#E63946', '#2196F3', '#4CAF50']
linestyles = ['-', '--', '-.']

for (label, res), color, ls in zip(results, colors, linestyles):
    axes[0].plot(res.best_fitness_hist, label=label, color=color, linestyle=ls, lw=2)
    axes[1].plot(res.avg_fitness_hist,  label=label, color=color, linestyle=ls, lw=2, alpha=0.8)

axes[0].set_title('Melhor Fitness por Geração', fontsize=12)
axes[0].set_xlabel('Geração')
axes[0].set_ylabel('Fitness (minimizar)')
axes[0].legend(fontsize=9)
axes[0].grid(True, alpha=0.3)

axes[1].set_title('Fitness Médio por Geração', fontsize=12)
axes[1].set_xlabel('Geração')
axes[1].set_ylabel('Fitness médio')
axes[1].legend(fontsize=9)
axes[1].grid(True, alpha=0.3)

plt.suptitle('Convergência do AG — Otimização de Rotas Saúde da Mulher', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(str(RESULTS_DIR / 'convergence.png'), dpi=150, bbox_inches='tight')
plt.show()
print('Gráfico salvo!')

In [ ]:
# Decomposição do fitness — melhor solução de cada experimento
fig, ax = plt.subplots(figsize=(12, 6))

labels_short = ['Exp 1\nPop=50', 'Exp 2\nPop=100', 'Exp 3\nPop=200']
components = ['distance', 'penalty_priority', 'penalty_capacity', 'penalty_time']
comp_labels = ['Distância', 'Pen. Prioridade', 'Pen. Capacidade', 'Pen. Tempo']
comp_colors = ['#3E8EDE', '#E63946', '#FF9800', '#9C27B0']

x = np.arange(len(labels_short))
width = 0.18

for i, (comp, clabel, color) in enumerate(zip(components, comp_labels, comp_colors)):
    values = [fitness_breakdown(res.best_individual.chromosome, points, dist_matrix)[comp]
              for _, res in results]
    ax.bar(x + i * width, values, width, label=clabel, color=color, alpha=0.85)

ax.set_xticks(x + width * 1.5)
ax.set_xticklabels(labels_short)
ax.set_title('Decomposição do Fitness — Melhor Solução por Experimento', fontsize=12)
ax.set_ylabel('Valor da componente')
ax.legend(fontsize=9)
ax.grid(True, axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig(str(RESULTS_DIR / 'fitness_breakdown.png'), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Identificar o melhor experimento geral
best_idx = min(range(len(results)), key=lambda i: results[i][1].best_individual.fitness)
best_label, best_result = results[best_idx]
best_route = best_result.best_individual.chromosome

print(f'Melhor experimento: {best_label}')
print(f'Fitness: {best_result.best_individual.fitness:.2f}')
print(f'Distância: {_route_distance(best_route, dist_matrix):.1f} km')
print(f'Rota: {best_route}')

In [ ]:
# Horário simulado da melhor rota
schedule = compute_arrival_times(best_route, points, dist_matrix)
point_map = {p.id: p for p in points}
from fase2_vrp.src.data_generator import TYPE_LABELS

print('\nHorário simulado da melhor rota:')
print(f'{"Parada":>6}  {"Ponto":<35} {"Chegada":>8}  {"Saída":>8}  {"Tipo"}')
print('-' * 85)
for i, (pid, arr, dep) in enumerate(schedule, 1):
    pt = point_map[pid]
    h_arr = int(arr); m_arr = int((arr - h_arr) * 60)
    h_dep = int(dep); m_dep = int((dep - h_dep) * 60)
    label = TYPE_LABELS[pt.type]
    tw_ok = '✓' if pt.time_window[0] <= arr <= pt.time_window[1] else '✗ FORA'
    print(f'{i:>6}.  {pt.name:<35} {h_arr:02d}:{m_arr:02d}  →  {h_dep:02d}:{m_dep:02d}  {label} {tw_ok}')

In [ ]:
# Gerar mapa interativo da melhor rota
map_path = str(RESULTS_DIR / 'route_map.html')
create_route_map(depot, points, best_route, dist_matrix, output_path=map_path)

# Gerar mapa de comparação dos 3 experimentos
comparison_data = [
    (label, res.best_individual.chromosome, res.best_individual.fitness)
    for label, res in results
]
comparison_path = str(RESULTS_DIR / 'comparison_map.html')
create_comparison_map(depot, points, comparison_data, dist_matrix, output_path=comparison_path)

print('\nMapas gerados! Abra os arquivos HTML no navegador.')
print(f'  - {map_path}')
print(f'  - {comparison_path}')

In [ ]:
# Salvar resultados em JSON
exp_data = []
for label, res in results:
    route = res.best_individual.chromosome
    bd = fitness_breakdown(route, points, dist_matrix)
    exp_data.append({
        'label': label,
        'config': {
            'population_size': res.config.population_size,
            'n_generations':   res.config.n_generations,
            'mutation_rate':   res.config.mutation_rate,
        },
        'best_fitness':    round(res.best_individual.fitness, 4),
        'initial_fitness': round(res.best_fitness_hist[0], 4),
        'improvement_pct': round((1 - res.best_individual.fitness / res.best_fitness_hist[0]) * 100, 2),
        'execution_time_s': res.execution_time_s,
        'best_route': route,
        'fitness_breakdown': bd,
    })

json_path = str(RESULTS_DIR / 'experiment_results.json')
with open(json_path, 'w', encoding='utf-8') as f:
    json.dump(exp_data, f, indent=2, ensure_ascii=False)

print(f'Resultados salvos em: {json_path}')
print('\n=== RESUMO FINAL ===')
display(df_results)

## Análise dos Resultados

### Comparação entre Experimentos

Os três experimentos demonstram o trade-off clássico do AG:

- **Exp 1 (Pop=50, Mut=0.10, 30 gen):** Convergência rápida, maior risco de ótimos locais
- **Exp 2 (Pop=100, Mut=0.05, 50 gen):** Equilíbrio entre exploração e explotação
- **Exp 3 (Pop=200, Mut=0.15, 50 gen):** Maior diversidade, busca mais ampla, maior custo computacional

### Impacto na Saúde da Mulher

A função fitness penaliza fortemente emergências obstétricas visitadas tarde:
- Penalidade de prioridade garante que emergências sejam atendidas no primeiro terço da rota
- Cada minuto de atraso em uma emergência obstétrica pode ser crítico
- A rota otimizada representa uma melhoria real no tempo de resposta